# 00 · Data: Sentinel-2 download, chips, Train/Val and Test folders
Rebuilds the full dataset: 104 Sentinel-2 images (13 UAE sites × 8 dates), the 256 × 256 chips, and the Train/Val and locked Test folders. Each long step skips itself when its output already exists.

## Environment
Runs in the `mangrove` conda env (`2_Code\environment.yml`; setup in `RUN_LOCALLY.md`).

## Paths
`paths.py` defines every folder in one place; Jupyter is launched from the SDP folder, so no paths need editing.
* `3_Data/`: small inputs (sites, labels, image list, folds)
* `6_Work/`: everything the pipeline rebuilds (images, chips, datasets), about 4 GB
* `4_Results/`: tables and figures

In [ ]:
import os, subprocess, sys
from pathlib import Path

for _d in [Path.cwd(), *Path.cwd().parents]:          # works from SDP, SDP/2_Code or SDP/2_Code/notebooks
    if (_d / 'paths.py').exists():
        CODE = _d; break
    if (_d / '2_Code' / 'paths.py').exists():
        CODE = _d / '2_Code'; break
else:
    raise FileNotFoundError('2_Code/paths.py not found: start Jupyter inside the SDP folder')
sys.path.insert(0, str(CODE))
import paths as P

PROJECT_ROOT, LAYOUT = P.PROJECT_ROOT, 'sdp'
WORK, SITES, LABELS, SCENES, FOLDS, QA_CSV = P.WORK, P.SITES, P.LABELS, P.SCENES, P.FOLDS, P.QA_CSV
DATASETS, RESULTS = P.DATASETS, P.RESULTS
TRAINVAL = P.TRAIN_VAL            # the Test_Dataset folder is never opened in these notebooks

def run_step(*args):
    """Run 2_Code/run_all.py with these arguments and show its output here, live."""
    env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([sys.executable, str(CODE / 'run_all.py'), *map(str, args)], cwd=PROJECT_ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError('run_all.py failed: see the messages above')

print(P.describe())

In [ ]:
import json, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 11})
pd.set_option('display.width', 160)

## Step 1 · Image list
`scenes_selected.csv` fixes the images used: 8 dates per site (four in 2020, March and November of 2018 and 2025), all under 10% cloud. A fixed list keeps the dataset reproducible.

In [ ]:
scenes = pd.read_csv(SCENES)
detail_path = SCENES.parent / 'scenes_selected_detail.csv'
detail = pd.read_csv(detail_path) if detail_path.exists() else None
print(f'{len(scenes)} image rows for {scenes.site.nunique()} sites')
if detail is not None:
    table = detail.drop_duplicates(['site', 'target']).pivot(index='site', columns='target', values='date')
    display(table)

## Step 2 · Download
`collect.py` pulls each scene from the Earth Search Sentinel-2 archive, clips it to the site, resamples to a 10 m grid and converts to reflectance (DN × 0.0001 − 0.1, applied once). About 2.4 GB into `6_Work/raw`, 10–20 min; existing files are skipped. Equivalent to `python 2_Code\run_all.py --step download`.

In [ ]:
manifest = P.RAW_MANIFEST
run_step('--step', 'download')          # quick trial instead: run_step('--step', 'download', '--sites', 'ghantoot', '--dates', '2020')
print(len(pd.read_csv(manifest)), 'images listed in', manifest)

## Step 3 · Coverage
Cloud and shadow cover (%) per site and date, taken from each scene's own mask (green = clear, red = cloudy). Check: no empty cells.

In [ ]:
m = pd.read_csv(manifest)
if detail is not None:
    m = m.merge(detail.drop_duplicates(['site', 'date'])[['site', 'date', 'target']], on=['site', 'date'], how='left')
    col = 'target'
else:
    col = 'date'
grid = m.pivot_table(index='site', columns=col, values='masked_fraction', aggfunc='max') * 100
fig, ax = plt.subplots(figsize=(10, 5.5))
im = ax.imshow(grid.values, cmap='RdYlGn_r', vmin=0, vmax=20, aspect='auto')
ax.set_xticks(range(grid.shape[1]), grid.columns, rotation=30, ha='right'); ax.set_yticks(range(grid.shape[0]), grid.index)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        v = grid.values[i, j]
        ax.text(j, i, '-' if np.isnan(v) else f'{v:.0f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, label='cloud / shadow over the site (%)'); ax.set_title('Download coverage: % of each site hidden by cloud')
plt.tight_layout(); plt.show()

## Step 4 · Quick look
True-colour view of the four Abu Dhabi training sites, November 2020. Mangroves show as dark green patches along the channels.

In [ ]:
import rasterio
show_sites = [s for s in ['eastern_mangroves', 'jubail_island', 'ras_ghurab', 'ghantoot'] if s in set(m.site)]   # those downloaded
fig, axes = plt.subplots(1, max(len(show_sites), 2), figsize=(16, 4.5))
for ax, site in zip(axes, tqdm(show_sites, desc='Reading images')):
    rows = m[(m.site == site) & (m.date.str.startswith('2020-11'))]
    row = rows.iloc[0] if len(rows) else m[m.site == site].iloc[0]
    with rasterio.open(WORK / row.path) as r:
        f = max(1, max(r.height, r.width) // 700)
        a = r.read([3, 2, 1], out_shape=(3, r.height // f, r.width // f))
    ax.imshow(np.dstack([np.clip(np.nan_to_num(b) / .25, 0, 1) for b in a]))
    ax.set_title(f'{site}\n{row.date}'); ax.axis('off')
plt.tight_layout(); plt.show()

## Step 5 · Chips
`prepare.py` cuts each image into 256 × 256 px chips (2.56 km). Each chip carries 6 bands, NDVI, MNDWI, both mangrove labels (GMW v4.0.19, 10 m, 2020; GMW v4.1.12, ~30 m, every year) and a cloud mask. Chips above 5% cloud are dropped, and only a subset of the chips with no mangrove is kept. About 3 min, into `6_Work/chips`; skipped if present.

In [ ]:
catalog_path = P.CATALOG
chips_dir = P.CHIPS
if catalog_path.exists() and chips_dir.exists() and any(chips_dir.glob('*.npz')):
    print('Chips already made - skipping.', len(pd.read_csv(catalog_path)), 'chips in', chips_dir)
else:
    run_step('--step', 'prepare')

## Step 6 · Chip counts
Chips per site and split. Test and external sites are only counted; their images stay unseen until the final evaluation.

In [ ]:
cat = pd.read_csv(catalog_path)
display(pd.crosstab(cat.split, cat.year, margins=True))
per_site = cat.groupby(['site', 'split']).size().reset_index(name='chips').sort_values(['split', 'chips'])
colors = {'train': '#2E5597', 'val': '#E8A33D', 'test': '#B03A2E', 'external': '#7B5EA7'}
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(per_site.site, per_site.chips, color=[colors.get(s, 'grey') for s in per_site.split])
for s, c in colors.items():
    ax.bar(0, 0, color=c, label=s)
ax.set_xlabel('chips (all dates)'); ax.set_title('Chips per site, coloured by the original split'); ax.legend()
plt.tight_layout(); plt.show()

## Step 7 · Land cover and datasets
* Land cover: ESA WorldCover class for every chip pixel, used by the baselines to see where errors fall (about 2 min).
* `Train_Val_Dataset`: the 6 Abu Dhabi cross-validation sites, 5 folds. All model development happens here.
* `Test_Dataset` (locked): the 2 fixed test sites and 5 external sites. Written here, never read.

Chips are hard-linked, so no extra disk space is used.

In [ ]:
run_step('--step', 'landcover', 'datasets')

tv = pd.read_csv(TRAINVAL / 'catalog.csv')
print('Train/Val chips:', len(tv), '| with the 10 m training label (2020):', int((tv.year == 2020).sum()))
display(pd.crosstab(tv.site, tv.year, margins=True))

**Done.** Next: `01_explore_data.ipynb`, which inspects the Train/Val chips before any model is trained.